<a href="https://colab.research.google.com/github/gyeongee/TIL/blob/main/%EB%AC%B8%EC%84%9C%EA%B5%B0%EC%A7%91%ED%99%94.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

문서 군집화(Document Clustering)

동일한 군집에 속하는 문서를 같은 카테고리 소속으로 분류합니다.

텍스트 분류 기반의 문서 분류는 사전에 결정 카테고리 값을 가진 학습 데이터 세트가 필요한 데 반해, 문서 군집화는 학습 데이터 세트가 필요 없는 비지도 학습 기반으로 동작합니다.

### Opinion Review 데이터 세트를 이용한 문서 군집화 수행하기

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [7]:
import zipfile
import os

zip_path = '/content/opinosis+opinion+frasl+review.zip'
extract_path = 'topics'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    for file in zip_ref.namelist():
        if '/topics' in file:
            zip_ref.extract(file, extract_path)

In [8]:
import glob,os
# 여러 개의 파일을 DataFrame으로 로딩
path= '/content/topics/OpinosisDataset1.0/topics'
# 모든 .data 파일들의 파일명을 리스트로 취합
all_files=glob.glob(os.path.join(path,"*.data"))
filename_li=[]
opinion_text=[]

# 개별 파일들의 파일명은 filename_li 리스트로 취합
# 개별 파일들의 파일 내용은 DataFrame 로딩 후 다시 string으로 변환하여 opinion_text 리스트로 취합
for file_ in all_files:
  # 개별 파일을 읽어서 DataFrame으로 생성
  df = pd.read_table(file_,index_col=None,header=0,encoding='latin1')
  # 절대경로로 주어진 파일명을 가공. Linux에서 수행 시에는 아래 \\를 / 변경
  filename_=file_.split('\\')[-1]
  filename=filename_.split('.')[0]

  # 파일명 리스트와 파일 내용 리스트에 파일명과 파일 내용을 추가
  filename_li.append(filename)
  opinion_text.append(df.to_string())

# 파일명 리스트와 파일 내용 리스트를 DataFrame으로 생성
document_df=pd.DataFrame({'filename':filename_li,'opinion_text':opinion_text})
document_df.head()


,filename,opinion_text
0,/content/topics/OpinosisDataset1,...
1,/content/topics/OpinosisDataset1,...
2,/content/topics/OpinosisDataset1,...
3,/content/topics/OpinosisDataset1,...
4,/content/topics/OpinosisDataset1,...


TfidfVectorizer는 Lemmatization과 같은 어근 변환을 직접 지원하진 않지만, tokenizer 인자에 커스텀 어근 변환 함수를 적용해 어근 변환을 수행할 수 있음

TfidfVectorizer의 tokenizer 인자로 사용될 lemmatization 어근 변환(기본형) 함수 설정

In [9]:
from nltk.stem import WordNetLemmatizer
import nltk
import string

# string.punctuation는 영어 구두점들을 가져옵니다.(예:!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~)
# ord() : 문자 하나를 해당 문자의 유니코드(Unicode)정수값으로 변환
# 문장부호 각각을 유니코드 정수값으로 변환하고, 그 값을 key로 하는 dictionary 만듦
remove_punct_dict = dict((ord(punct),None)for punct in string.punctuation)
lemmar=WordNetLemmatizer()

# 입력으로 들어온 token단어들에 대해서 lemmatization 어근 변환
def LemTokens(tokens):
  return [lemmar.lemmatize(token)for token in tokens]

# TfidfVectorizer 객체 생성 시 tokenizer 인자로 해당 함수를 설정하여 lemmatization 적용
# 입력으로 문장을 받아서 stop words 제거 -> 소문자 변환 -> 단어 토큰화 -> lemmatization 어근 변환
def LemNormalize(text):
  return LemTokens(nltk.word_tokenize(text.lower().translate(remove_punct_dict)))

개별 문서 텍스트에 대해 TF-IDF 변환된 피처 벡터화

In [10]:
# nltk 데이터 다운
import nltk

nltk.download('punkt_tab')
nltk.download('wordnet')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vect = TfidfVectorizer(tokenizer=LemNormalize,stop_words='english',
                             ngram_range=(1,2),min_df=0.05,max_df=0.85)

#opinion_text 칼럼값으로 feature vectorization 수행
feature_vect = tfidf_vect.fit_transform(document_df['opinion_text'])

In [12]:
# 문서별 텍스트가 TF-IDF 변환된 피처 벡터화 행렬 데이터에 대해서 군집화를 수행해 어떤 문서끼리 군집되는지 확인
# 5개의 중심(Centroid)
from sklearn.cluster import KMeans

# 5개 집합으로 군집화 수행, 예제를 위해 동일한 클러스터링 결과 도출용 random_state=0
km_cluster=KMeans(n_clusters=5,max_iter=10000,random_state=0)
km_cluster.fit(feature_vect)
cluster_label=km_cluster.labels_
cluster_centers=km_cluster.cluster_centers_

In [13]:
document_df['cluster_label']=cluster_label
document_df.head()

,filename,opinion_text,cluster_label
0,/content/topics/OpinosisDataset1,...,4
1,/content/topics/OpinosisDataset1,...,4
2,/content/topics/OpinosisDataset1,...,3
3,/content/topics/OpinosisDataset1,...,1
4,/content/topics/OpinosisDataset1,...,4


In [14]:
# Cluster #0은 호텔에 대한 리뷰로 군집화돼 있습니다.
document_df[document_df['cluster_label']==0].sort_values(by='filename')

,filename,opinion_text,cluster_label
25,/content/topics/OpinosisDataset1,...,0
27,/content/topics/OpinosisDataset1,...,0


전반적으로 군집화된 결과를 살펴보면 군집 개수가 약간 많게 설정돼 있어서 세분화되어 군집화된 경향이 있습니다.

In [15]:
# cluster=3
from sklearn.cluster import KMeans

# 3개의 집합으로 군집화
km_cluster=KMeans(n_clusters=3,max_iter=10000,random_state=0)
km_cluster.fit(feature_vect)
cluster_label=km_cluster.labels_

# 소속 클러스터 cluster_label 칼럼으로 할당하고 cluster_label 값으로 정렬
document_df['cluster_label']=cluster_label
document_df.sort_values(by='cluster_label')

,filename,opinion_text,cluster_label
0,/content/topics/OpinosisDataset1,...,0
1,/content/topics/OpinosisDataset1,...,0
4,/content/topics/OpinosisDataset1,...,0
9,/content/topics/OpinosisDataset1,...,0
13,/content/topics/OpinosisDataset1,...,0
22,/content/topics/OpinosisDataset1,...,0
20,/content/topics/OpinosisDataset1,...,0
17,/content/topics/OpinosisDataset1,...,0
25,/content/topics/OpinosisDataset1,...,0
27,/content/topics/OpinosisDataset1,...,0


군집화가 잘 된것을 알 수 있습니다

### 군집별 핵심 단어 추출하기

각 군집(cluster)에 속한 문서는 핵심 단어를 주축으로 군집화돼 있을 것입니다.

이번에는 각 군집을 구성하는 핵심 단어가 어떤 것이 있는지 확인해 보겠습니다.

KMeans의 `cluster_centers_[0,1]`은 0번 군집에서 두 번째 피처의 값으로 표현한 일종의 좌표값입니다.

In [17]:
cluster_centers=km_cluster.cluster_centers_
print(f'cluster_centers shape:{cluster_centers.shape}')
print(cluster_centers)

cluster_centers shape:(3, 4610)
[[0.         0.00099548 0.00174656 ... 0.         0.00183397 0.00144581]
 [0.00634592 0.00057845 0.         ... 0.00052665 0.         0.        ]
 [0.00788567 0.         0.         ... 0.00884976 0.         0.        ]]


각 행의 배열 값은 각 군집 내의 4611개 피처의 위치가 개별 중심과 얼마나 가까운가를 상대 값으로 나타낸 것입니다.

0에서 1까지의 값을 가질 수 있으며 1에 가까울수록 중심과 가까운 값을 의미합니다.



In [29]:
# 각 군집별 핵심 단어 찾기
# 군집별 top n 핵심 단어, 그 단어의 중심 위치 상댓값, 대상 파일명을 반환함
def get_cluster_details(cluster_model,cluster_data,feature_names,cluster_num,top_n_features=10):
  cluster_details={}

  # cluster_centers array의 값이 큰 순으로 정렬된 인덱스 값을 반환
  # 군집 중심점(centroid)별 할당된 word 피처들의 거리값이 큰 순으로 값을 구하기 위함
  centroid_feature_ordered_ind = cluster_model.cluster_centers_.argsort()[:,::-1]

  # 개별 군집별로 반복하면서 핵심 단어, 그 단어의 중심 위치 상댓값, 대상 파일명 입력
  for cluster_num in range(cluster_num):
    # 개별 군집별 정보를 담을 데이터 초기화
    cluster_details[cluster_num]={}
    cluster_details[cluster_num]['cluster']=cluster_num

    #cluster_centers_.argsort()[:,::-1]로 구한 인덱스를 이용해 top n 피처 단어를 구합니다.
    top_feature_indexes = centroid_feature_ordered_ind[cluster_num,:top_n_features]
    top_features=[feature_names[ind] for ind in top_feature_indexes]

    # top_feature_indexes를 이용해 해당 피처 단어의 중심 위치 상댓값을 구함
    top_feature_values=cluster_model.cluster_centers_[cluster_num,
                                                      top_feature_indexes].tolist()

    # clsuter_details 딕셔너리 객체에 개별 군집별 핵심단어와 중심 위치 상댓값,해당 파일명 입력
    cluster_details[cluster_num]['top_features']=top_features
    cluster_details[cluster_num]['top_features_value']=top_feature_values
    filenames=cluster_data[cluster_data['cluster_label']==cluster_num]['filename']
    filenames=filenames.values.tolist()

    cluster_details[cluster_num]['filenames']=filenames

  return cluster_details

`get_cluster_details()`를 호출하면 dictionary를 원소로 가지는 리스트인 cluster_details를 반환합니다.

In [32]:
def print_clsuter_details(cluster_details):
  for cluster_num,cluster_detail in cluster_details.items():
    print(f'#### Cluster {cluster_num}')
    print(f'Top features:{cluster_detail['top_features']}')
    print(f'Reviwes 파일명:{cluster_detail['filenames'][:7]}')
    print('##################################################')

In [33]:
feature_names=tfidf_vect.get_feature_names_out()

cluster_details=get_cluster_details(cluster_model=km_cluster,cluster_data=document_df,
                                    feature_names=feature_names,cluster_num=3,top_n_features=10)
print_clsuter_details(cluster_details)


#### Cluster 0
Top features:['room', 'hotel', 'service', 'staff', 'food', 'location', 'bathroom', 'clean', 'price', 'parking']
Reviwes 파일명:['/content/topics/OpinosisDataset1', '/content/topics/OpinosisDataset1', '/content/topics/OpinosisDataset1', '/content/topics/OpinosisDataset1', '/content/topics/OpinosisDataset1', '/content/topics/OpinosisDataset1', '/content/topics/OpinosisDataset1']
##################################################
#### Cluster 1
Top features:['interior', 'seat', 'performance', 'mileage', 'comfortable', 'keyboard', 'gas', 'quality', 'gas mileage', 'transmission']
Reviwes 파일명:['/content/topics/OpinosisDataset1', '/content/topics/OpinosisDataset1', '/content/topics/OpinosisDataset1', '/content/topics/OpinosisDataset1', '/content/topics/OpinosisDataset1', '/content/topics/OpinosisDataset1', '/content/topics/OpinosisDataset1']
##################################################
#### Cluster 2
Top features:['screen', 'battery', 'direction', 'voice', 'map', 'feature', 